# CNN - Reconhecimento de incêndio em imagens RGB (fire / nofire)

Notebook de experimentação da parte de **Deep Learning** do projeto IAA3 (seção 14 do PDF da atividade).
Ele chama as funções de `src/` (o código real fica lá) e mostra os resultados. **As saídas só aparecem depois de
você executar as células** (é necessário TensorFlow: `pip install -r requirements.txt`).

In [ ]:
import sys, json
from pathlib import Path

# acha a raiz do projeto, mesmo se o notebook for aberto de dentro de notebooks/
RAIZ = Path.cwd()
if not (RAIZ / "src").is_dir():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
print("Raiz do projeto:", RAIZ)

## 1. Dataset, classes e divisão treino/validação/teste
`fire` = 1 (incêndio) e `nofire` = 0. A divisão (70/15/15) é estratificada e agrupa imagens quase idênticas
para evitar vazamento entre os conjuntos.

In [ ]:
import cnn_dados as dados

manifesto = dados.construir_manifesto()          # lê as imagens, agrupa duplicatas e divide (com verificação)
print(json.dumps(dados.resumo_dataset(manifesto), indent=2, ensure_ascii=False))
manifesto.head()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

amostra = manifesto[manifesto["conjunto"] == "train"].groupby("classe").head(4)
fig, eixos = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, r) in zip(eixos.ravel(), amostra.sort_values("classe").iterrows()):
    ax.imshow(Image.open(dados.DATA_DIR / r["arquivo"]).convert("RGB"))
    ax.set_title(f"{r['classe']} ({r['arquivo'].split('/')[-1]})", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Treinamento e avaliação
Roda o pipeline completo (treino, salvamento, recarga do modelo, avaliação no teste e análises).
Para um teste rápido troque por `treinar(smoke=True, epocas=2, resultados_dir=..., modelo_saida=...)`
ou use `python src/cnn_treino.py --smoke` no terminal. Pode levar vários minutos em CPU.

In [ ]:
import cnn_treino

resultado = cnn_treino.treinar(epocas=40)        # hiperparâmetros: epocas, batch, tamanho, paciencia, limiar, aumento

## 3. Resultados (arquivos gerados em `results/cnn/`)

In [ ]:
from IPython.display import Image as IImage, Markdown, display

pasta = RAIZ / "results" / "cnn"
m = json.loads((pasta / "metricas_cnn.json").read_text(encoding="utf-8"))
print({k: round(m[k], 4) for k in ("accuracy", "precision", "recall", "f1_score",
                                  "taxa_falsos_positivos", "taxa_falsos_negativos")})
print("Matriz de confusão [nofire, fire]:", m["matriz_confusao"])
for arq in ("matriz_confusao.png", "historico_treinamento.png", "analise_generalizacao.png"):
    display(IImage(filename=str(pasta / arq)))

### Exemplos de acertos e de erros (imagens reais do conjunto de teste)

In [ ]:
display(IImage(filename=str(pasta / "exemplos_acertos.png")))
display(IImage(filename=str(pasta / "exemplos_erros.png")))

In [ ]:
display(Markdown((pasta / "relatorio_cnn.md").read_text(encoding="utf-8")))

## 4. Prever uma imagem nova
Troque o caminho pela sua imagem.

In [ ]:
import cnn_prever

exemplo = dados.DATA_DIR / "fire" / "fire_0001.jpg"   # substitua por uma imagem sua
print(cnn_prever.prever_imagem(exemplo))

## 5. Ablação opcional: sem aumento de dados
Compare o gap treino x teste (`analise_generalizacao.json`) com e sem aumento de dados:

```
python src/cnn_treino.py --sem-aumento --modelo-saida models/cnn_sem_aumento.keras --resultados-dir results/cnn_sem_aumento
```